In [58]:
import copy

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
  accuracy_score,
  f1_score,
  precision_score,
  recall_score,
)
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
import joblib

In [22]:
def load_data(file_path):
    data = pd.read_csv(file_path)
    print(f"Loaded {len(data)} rows from {file_path}")  
    print(data[data["review_text"].isna()])

    data = data.dropna(subset=["review_text"])

    # Keep relevant columns and drop NaNs
    data = data[["review_text", "Final Sentiment"]].dropna()

    data = data.dropna(subset=["Final Sentiment"])

    return data


In [9]:
def creating_vectorizers(data):
    """
    This function creates and returns different types of vectorizers for the given data.
    It uses CountVectorizer and TfidfVectorizer from sklearn.
    """
    # Count Vectorizer
    unigram_count_vectorizer = CountVectorizer(ngram_range=(1, 1))  # Use unigrams
    X_uni_count = unigram_count_vectorizer.fit_transform(data["review_text"])
    print("Count Vectorizer Shape:", X_uni_count.shape)

    bigram_count_vectorizer = CountVectorizer(
        ngram_range=(1, 2)
    )  # Use unigrams and bigrams
    X_bi_count = bigram_count_vectorizer.fit_transform(
        data["review_text"]
    )  # Convert text to numerical features
    print("Count Vectorizer Shape:", X_bi_count.shape)

    # TF-IDF Vectorizer
    unigram_tfidf_vectorizer = TfidfVectorizer(
        ngram_range=(1, 1)
    )  # Use unigrams and bigrams
    X_uni_tfidf = unigram_tfidf_vectorizer.fit_transform(data["review_text"])
    print("TF-IDF Vectorizer Shape:", X_uni_tfidf.shape)

    bigram_tfidf_vectorizer = TfidfVectorizer(
        ngram_range=(1, 2)
    )  # Use unigrams and bigrams
    X_bi_tfidf = bigram_tfidf_vectorizer.fit_transform(data["review_text"])
    print("TF-IDF Vectorizer Shape:", X_bi_tfidf.shape)

    return {
        "Unigram Count Vec": X_uni_count,
        "Bigram Count Vec": X_bi_count,
        "Unigram TF-IDF Vec": X_uni_tfidf,
        "Bigram TF-IDF Vec": X_bi_tfidf,
    }

## Subjectivity Detection

In [10]:
def subjectivity_detection(data, models):
    """
    This function performs subjectivity detection on the given text based on the Final Sentiment.
    It creates a new column 'subjectivity' and performs classification to predict whether
    the sentiment is subjective (positive/negative) or objective (neutral).
    """
    vectorizers = creating_vectorizers(data)

    results_unigram = []
    results_bigram = []

    best_score = 0.0
    best_combo = {"model_name": None, "vectorizer": None, "model": None, "accuracy": 0.0}

    for model_name, base_model in models.items():
        print(f"\n=== Using model: {model_name} ===")

        for vectorizer_name, X in vectorizers.items():
            print(f"\n--- Vectorizer: {vectorizer_name} ---")

            X_train, X_test, y_train, y_test = train_test_split(
                X, data["subjectivity"], test_size=0.2, random_state=42
            )

            # Initialize K-Fold
            kf = KFold(n_splits=5, shuffle=True, random_state=42)

            fold_accuracies = []
            fold_precisions = []
            fold_recalls = []
            fold_f1_scores = []

            for train_idx, val_idx in kf.split(X_train):
                X_train_fold = X_train[train_idx]
                X_val_fold = X_train[val_idx]
                y_train_fold = y_train.iloc[train_idx]
                y_val_fold = y_train.iloc[val_idx]

                # Make a fresh copy of the model for each fold
                current_model = copy.deepcopy(base_model)

                # Train the model
                current_model.fit(X_train_fold, y_train_fold)

                # If GridSearchCV, use best_estimator_ for prediction
                if isinstance(current_model, GridSearchCV):
                    y_pred_val = current_model.best_estimator_.predict(X_val_fold)
                else:
                    y_pred_val = current_model.predict(X_val_fold)

                # Calculate metrics
                fold_accuracies.append(accuracy_score(y_val_fold, y_pred_val))
                fold_precisions.append(precision_score(y_val_fold, y_pred_val, average='binary', pos_label='Subjective'))
                fold_recalls.append(recall_score(y_val_fold, y_pred_val, average='binary', pos_label='Subjective'))
                fold_f1_scores.append(f1_score(y_val_fold, y_pred_val, average='binary', pos_label='Subjective'))

            # Average K-Fold metrics
            k_acc = np.mean(fold_accuracies)
            k_pre = np.mean(fold_precisions)
            k_rec = np.mean(fold_recalls)
            k_f1 = np.mean(fold_f1_scores)

            # Now retrain on full training set
            final_model = copy.deepcopy(base_model)
            final_model.fit(X_train, y_train)

            if isinstance(final_model, GridSearchCV):
                best_model = final_model.best_estimator_
            else:
                best_model = final_model

            y_pred_test = best_model.predict(X_test)

            test_accuracy = accuracy_score(y_test, y_pred_test)
            test_precision = precision_score(y_test, y_pred_test, average='binary', pos_label='Subjective')
            test_recall = recall_score(y_test, y_pred_test, average='binary', pos_label='Subjective')
            test_f1 = f1_score(y_test, y_pred_test, average='binary', pos_label='Subjective')

            # Update best combo if needed
            if test_accuracy > best_score:
                best_score = test_accuracy
                best_combo = {
                    "model_name": model_name,
                    "vectorizer": vectorizer_name,
                    "model": best_model,
                    "accuracy": best_score
                }

            # Store results
            row = [model_name + " + " + vectorizer_name, k_acc, k_pre, k_rec, k_f1, test_accuracy, test_precision, test_recall, test_f1]
            if "unigram" in vectorizer_name.lower():
                results_unigram.append(row)
            elif "bigram" in vectorizer_name.lower():
                results_bigram.append(row)

    # Prepare DataFrames
    columns = ["Model", "K-Acc", "K-Pre", "K-Rec", "K-F1", "Test-Acc", "Test-Pre", "Test-Rec", "Test-F1"]

    unigram_df = pd.DataFrame(results_unigram, columns=columns)
    bigram_df = pd.DataFrame(results_bigram, columns=columns)

    # Print Results
    print("\n=== Unigram Results ===")
    print(unigram_df)

    print("\n=== Bigram Results ===")
    print(bigram_df)

    # Top 3
    top_models_unigram = unigram_df.sort_values(by="Test-Acc", ascending=False).head(3)
    top_models_bigram = bigram_df.sort_values(by="Test-Acc", ascending=False).head(3)

    print("\n=== Top 3 Models (Unigram) ===")
    print(top_models_unigram[["Model", "Test-Acc"]])

    print("\n=== Top 3 Models (Bigram) ===")
    print(top_models_bigram[["Model", "Test-Acc"]])

    top_models_combined = pd.concat([top_models_unigram, top_models_bigram]).sort_values(by="Test-Acc", ascending=False).head(3)
    print("\n=== Top 3 Models Overall ===")
    print(top_models_combined[["Model", "Test-Acc"]])

    return best_combo


In [24]:
# Load manually labeled data
datasets = [
    "annotated_microtext_lemma.csv",
]

models = {
    "Logistic Regression": LogisticRegression(solver="lbfgs", max_iter=1000),
    "SVM": SVC(kernel="rbf"),
    "Naive Bayes": MultinomialNB(),
    "Random Forest": RandomForestClassifier(random_state=42),
}

for dataset in datasets:
    print(f"\n=== Processing Dataset: {dataset} ===")

    data = load_data(dataset)

    print(f"Loaded {len(data)} reviews from {dataset}.")
    
    # Create a new column 'subjectivity' based on the 'Final Sentiment'
    data['subjectivity'] = data['Final Sentiment'].apply(
        lambda x: 'Subjective' if x in ['positive', 'negative'] else 'Objective'
    )
    
    print(f"Subjectivity Detection: {data['subjectivity'].value_counts()}")
    
    # Perform Subjectivity Detection first
    subjectivity_detection(data,models)



=== Processing Dataset: annotated_microtext_lemma.csv ===
Loaded 1249 rows from annotated_microtext_lemma.csv
Empty DataFrame
Columns: [Platform, Game, review_text, label1, label2, label3, Final Sentiment]
Index: []
Loaded 1249 reviews from annotated_microtext_lemma.csv.
Subjectivity Detection: subjectivity
Subjective    829
Objective     420
Name: count, dtype: int64
Count Vectorizer Shape: (1249, 8982)
Count Vectorizer Shape: (1249, 63169)
TF-IDF Vectorizer Shape: (1249, 8982)
TF-IDF Vectorizer Shape: (1249, 63169)

=== Using model: Logistic Regression ===

--- Vectorizer: Unigram Count Vec ---

--- Vectorizer: Bigram Count Vec ---

--- Vectorizer: Unigram TF-IDF Vec ---

--- Vectorizer: Bigram TF-IDF Vec ---

=== Using model: SVM ===

--- Vectorizer: Unigram Count Vec ---

--- Vectorizer: Bigram Count Vec ---

--- Vectorizer: Unigram TF-IDF Vec ---

--- Vectorizer: Bigram TF-IDF Vec ---

=== Using model: Naive Bayes ===

--- Vectorizer: Unigram Count Vec ---

--- Vectorizer: Bigram

In [ ]:
from sklearn.ensemble import VotingClassifier


def subjectivity_detection_ensemble(data, models, vectorizers):
    """
    Perform polarity detection using a VotingClassifier ensemble.
    
    Parameters:
    - data: DataFrame containing 'Final Sentiment' labels.
    - models: Dictionary of {model_name: model_instance}.
    - vectorizers: Dictionary of {vectorizer_name: vectorized features}.
    - voting_type: 'hard' or 'soft' voting.
    """
    results = []
    best_score = 0.0
    best_combo = {"model_name": None, "vectorizer": None, "model": None, "accuracy": 0.0}

    # Create the VotingClassifier from the models
    estimators = [(name, model) for name, model in models.items()]
    voting_clf = VotingClassifier(estimators=estimators, voting="hard")

    for vectorizer_name, X in vectorizers.items():
        print(f"\n--- Vectorizer: {vectorizer_name} ---")

        X_train, X_test, y_train, y_test = train_test_split(
            X, data["subjectivity"], test_size=0.2, random_state=42
        )

        current_model = copy.deepcopy(voting_clf)

        # Cross-validation
        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        fold_accuracies, fold_precisions, fold_recalls, fold_f1_scores = [], [], [], []

        for train_idx, val_idx in kf.split(X_train):
            X_train_fold, X_val_fold = X_train[train_idx], X_train[val_idx]
            y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]

            current_model.fit(X_train_fold, y_train_fold)
            y_pred_val = current_model.predict(X_val_fold)

            fold_accuracies.append(accuracy_score(y_val_fold, y_pred_val))
            fold_precisions.append(precision_score(y_val_fold, y_pred_val, average='binary', pos_label='Subjective'))
            fold_recalls.append(recall_score(y_val_fold, y_pred_val, average='binary', pos_label='Subjective'))
            fold_f1_scores.append(f1_score(y_val_fold, y_pred_val, average='binary', pos_label='Subjective'))

        # Average cross-validation metrics
        k_acc, k_pre, k_rec, k_f1 = map(np.mean, (fold_accuracies, fold_precisions, fold_recalls, fold_f1_scores))

        # Train on full train set and evaluate on test set
        current_model.fit(X_train, y_train)
        y_pred_test = current_model.predict(X_test)

        test_accuracy = accuracy_score(y_test, y_pred_test)
        test_precision = precision_score(y_test, y_pred_test, average='binary', pos_label='Subjective')
        test_recall = recall_score(y_test, y_pred_test, average='binary', pos_label='Subjective')
        test_f1 = f1_score(y_test, y_pred_test, average='binary', pos_label='Subjective')

        results.append([
            f"VotingClassifier + {vectorizer_name}",
            k_acc, k_pre, k_rec, k_f1,
            test_accuracy, test_precision, test_recall, test_f1
        ])

        if test_accuracy > best_score:
            best_score = test_accuracy
            best_combo["model_name"] = "VotingClassifier"
            best_combo["vectorizer"] = vectorizer_name
            best_combo["model"] = current_model
            best_combo["accuracy"] = best_score

    # Final results
    columns = ["Model", "K-Acc", "K-Pre", "K-Rec", "K-F1", "Test-Acc", "Test-Pre", "Test-Rec", "Test-F1"]
    results_df = pd.DataFrame(results, columns=columns)

    print("\n=== polarity detection Results ===")
    print(results_df)

    return best_combo

In [28]:
# Load manually labeled data
datasets = [
    "annotated_microtext_pos_lemma.csv"
]

models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(kernel="linear"),
    "Logistic Regression": LogisticRegression(solver="lbfgs", max_iter=1000),
}

vectorizers = creating_vectorizers(data)

for dataset in datasets:
    print(f"\n=== Processing Dataset: {dataset} ===")

    data = load_data(dataset)

    print(f"Loaded {len(data)} reviews from {dataset}.")
    
    # Create a new column 'subjectivity' based on the 'Final Sentiment'
    data['subjectivity'] = data['Final Sentiment'].apply(
        lambda x: 'Subjective' if x in ['positive', 'negative'] else 'Objective'
    )
    
    print(f"Subjectivity Detection: {data['subjectivity'].value_counts()}")
    
    # Perform Subjectivity Detection first
    subjectivity_detection_ensemble(data,models, vectorizers)


Count Vectorizer Shape: (1248, 8381)
Count Vectorizer Shape: (1248, 61595)
TF-IDF Vectorizer Shape: (1248, 8381)
TF-IDF Vectorizer Shape: (1248, 61595)

=== Processing Dataset: annotated_microtext_pos_lemma.csv ===
Loaded 1249 rows from annotated_microtext_pos_lemma.csv
       Platform            Game review_text   label1   label2   label3  \
938  Metacritic  Cyberpunk 2077         NaN  neutral  neutral  neutral   

    Final Sentiment  
938         neutral  
Loaded 1248 reviews from annotated_microtext_pos_lemma.csv.
Subjectivity Detection: subjectivity
Subjective    829
Objective     419
Name: count, dtype: int64

--- Vectorizer: Unigram Count Vec ---

--- Vectorizer: Bigram Count Vec ---

--- Vectorizer: Unigram TF-IDF Vec ---

--- Vectorizer: Bigram TF-IDF Vec ---

=== Sentiment Analysis Results ===
                                   Model     K-Acc     K-Pre     K-Rec  \
0   VotingClassifier + Unigram Count Vec  0.684347  0.728408  0.833040   
1    VotingClassifier + Bigram Count 

In [32]:
def subjectivity_detection_gridsearch(data, param_grid):
    """
    This function performs subjectivity detection using GridSearchCV for RandomForestClassifier.
    It skips manual K-Fold and focuses only on finding the best hyperparameters.
    """

    # Assuming creating_vectorizers is defined elsewhere
    vectorizers = creating_vectorizers(data)

    best_score = 0.0
    best_combo = {"model": None, "vectorizer": None, "accuracy": 0.0, "best_params": None}
    
    for vectorizer_name, X in vectorizers.items():
        print(f"\n--- Vectorizer: {vectorizer_name} ---")
        
        X_train, X_test, y_train, y_test = train_test_split(
            X, data["subjectivity"], test_size=0.2, random_state=42
        )

        # Define the GridSearchCV with Logistic Regression
        grid_search = GridSearchCV(
            estimator=LogisticRegression(random_state=42, max_iter=10000),
            param_grid=param_grid,
            cv=5,  # Usually GridSearchCV needs cv>1, so use 5-fold internally
            n_jobs=-1,
            verbose=1
        )

        # Fit GridSearchCV
        grid_search.fit(X_train, y_train)

        best_model = grid_search.best_estimator_

        # Predict on the test set
        y_pred_test = best_model.predict(X_test)

        # Calculate test metrics
        test_accuracy = accuracy_score(y_test, y_pred_test)
        test_precision = precision_score(y_test, y_pred_test, average='binary', pos_label='Subjective')
        test_recall = recall_score(y_test, y_pred_test, average='binary', pos_label='Subjective')
        test_f1 = f1_score(y_test, y_pred_test, average='binary', pos_label='Subjective')

        print(f"Test Accuracy: {test_accuracy:.4f}")
        print(f"Test Precision: {test_precision:.4f}")
        print(f"Test Recall: {test_recall:.4f}")
        print(f"Test F1 Score: {test_f1:.4f}")
        print(f"Best Params: {grid_search.best_params_}")

        # Update best combo if this model is better
        if test_accuracy > best_score:
            best_score = test_accuracy
            best_combo["model"] = best_model
            best_combo["vectorizer"] = vectorizer_name
            best_combo["accuracy"] = best_score
            best_combo["best_params"] = grid_search.best_params_

    return best_combo


In [33]:
# Load manually labeled data
datasets = [
    "annotated_microtext_pos_lemma.csv"
]

# Grid Search for Hyperparameter Tuning
param_grid = {
    'C': [0.1, 1, 10],
    'penalty': ['l1', 'l2'],  # or 'elasticnet', 'none'
    'solver': ['liblinear', 'saga'],  # 'liblinear' for small datasets, 'saga' for large datasets
}

for dataset in datasets:
    print(f"\n=== Processing Dataset: {dataset} ===")

    data = load_data(dataset)

    print(f"Loaded {len(data)} reviews from {dataset}.")
    
    # Create a new column 'subjectivity' based on the 'Final Sentiment'
    data['subjectivity'] = data['Final Sentiment'].apply(
        lambda x: 'Subjective' if x in ['positive', 'negative'] else 'Objective'
    )
    
    print(f"Subjectivity Detection: {data['subjectivity'].value_counts()}")
    
    # Perform Subjectivity Detection first
    subjectivity_detection_gridsearch(data,param_grid)



=== Processing Dataset: annotated_microtext_pos_lemma.csv ===
Loaded 1249 rows from annotated_microtext_pos_lemma.csv
       Platform            Game review_text   label1   label2   label3  \
938  Metacritic  Cyberpunk 2077         NaN  neutral  neutral  neutral   

    Final Sentiment  
938         neutral  
Loaded 1248 reviews from annotated_microtext_pos_lemma.csv.
Subjectivity Detection: subjectivity
Subjective    829
Objective     419
Name: count, dtype: int64
Count Vectorizer Shape: (1248, 8381)
Count Vectorizer Shape: (1248, 61595)
TF-IDF Vectorizer Shape: (1248, 8381)
TF-IDF Vectorizer Shape: (1248, 61595)

--- Vectorizer: Unigram Count Vec ---
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Test Accuracy: 0.6480
Test Precision: 0.7143
Test Recall: 0.8140
Test F1 Score: 0.7609
Best Params: {'C': 0.1, 'penalty': 'l2', 'solver': 'saga'}

--- Vectorizer: Bigram Count Vec ---
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Test Accuracy: 0.6600
Test Preci

/Users/bryan/GitHub/SC4021-Project/.venv/lib/python3.10/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


## Polarity Detection

In [ ]:
def polarity_detection(data, models):
    """
    This function performs polarity detection on the given text using TextBlob and SenticNet.
    It returns the sentiment polarity and subjectivity.
    """
    vectorizers = creating_vectorizers(data)
    
    # Store results for each model-vectorizer combination
    results_unigram = []
    results_bigram = []
    
    best_score = 0.0
    best_combo = {"model": None, "vectorizer": None, "accuracy": 0.0}
    for model_name, model in models.items():
        print(f"\n=== Using model: {model_name} ===")

        for vectorizer_name, X in vectorizers.items():
            print(f"\n--- Vectorizer: {vectorizer_name} ---")
            
            X_train, X_test, y_train, y_test = train_test_split(
                X, data["Final Sentiment"], test_size=0.2, random_state=42
            )

            # Make a fresh copy of the model so we don't mess up the original
            current_model = copy.deepcopy(model)

            # Perform K-Fold Cross-Validation
            kf = KFold(n_splits=5, shuffle=True, random_state=42)
            fold_accuracies = []
            fold_precisions = []
            fold_recalls = []
            fold_f1_scores = []
            
            for train_idx, val_idx in kf.split(X_train):
                X_train_fold, X_val_fold = X_train[train_idx], X_train[val_idx]
                y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]
                
                # Train the model on the fold
                current_model.fit(X_train_fold, y_train_fold)
                
                # If it's a GridSearchCV, update the model to the best estimator
                if isinstance(current_model, GridSearchCV):
                    current_model = current_model.best_estimator_

                # Predict on validation set
                y_pred_val = current_model.predict(X_val_fold)
                
                # Calculate metrics for the current fold
                fold_accuracies.append(accuracy_score(y_val_fold, y_pred_val))
                fold_precisions.append(precision_score(y_val_fold, y_pred_val, average='binary', pos_label='positive'))
                fold_recalls.append(recall_score(y_val_fold, y_pred_val, average='binary', pos_label='positive'))
                fold_f1_scores.append(f1_score(y_val_fold, y_pred_val, average='binary', pos_label='positive'))
            
            # Store K-Fold metrics for later
            k_acc = np.mean(fold_accuracies)
            k_pre = np.mean(fold_precisions)
            k_rec = np.mean(fold_recalls)
            k_f1 = np.mean(fold_f1_scores)
            
            # Store results for Unigrams or Bigrams
            if "unigram" in vectorizer_name.lower():
                results_unigram.append([model_name+" + "+vectorizer_name, k_acc, k_pre, k_rec, k_f1])
            elif "bigram" in vectorizer_name.lower():
                results_bigram.append([model_name+" + "+vectorizer_name, k_acc, k_pre, k_rec, k_f1])
            
            # Evaluate on the test set after training on the full train set
            current_model.fit(X_train, y_train)
            
            # If it's a GridSearchCV, update the model to the best estimator again
            if isinstance(current_model, GridSearchCV):
                current_model = current_model.best_estimator_

            y_pred_test = current_model.predict(X_test)

            # Calculate test metrics
            test_accuracy = accuracy_score(y_test, y_pred_test)
            test_precision = precision_score(y_test, y_pred_test, average='binary', pos_label='positive')
            test_recall = recall_score(y_test, y_pred_test, average='binary', pos_label='positive')
            test_f1 = f1_score(y_test, y_pred_test, average='binary', pos_label='positive')

            # Update best combination if this is better
            if test_accuracy > best_score:
                best_score = test_accuracy
                best_combo["model"] = model_name
                best_combo["vectorizer"] = vectorizer_name
                best_combo["accuracy"] = best_score
                
            # Store test metrics
            if "unigram" in vectorizer_name.lower():
                results_unigram[-1].extend([test_accuracy, test_precision, test_recall, test_f1])
            elif "bigram" in vectorizer_name.lower():
                results_bigram[-1].extend([test_accuracy, test_precision, test_recall, test_f1])
            
    # Convert results to DataFrame for easy printing
    columns = ["Model", "K-Acc", "K-Pre", "K-Rec", "K-F1", "Test-Acc", "Test-Pre", "Test-Rec", "Test-F1"]
    
    unigram_df = pd.DataFrame(results_unigram, columns=columns)
    bigram_df = pd.DataFrame(results_bigram, columns=columns)

    # Print the tables
    print("\n=== Unigram Results ===")
    print(unigram_df)
    
    print("\n=== Bigram Results ===")
    print(bigram_df)

    # Sort models based on Test-Accuracy
    top_models_unigram = unigram_df.sort_values(by="Test-Acc", ascending=False).head(3)
    top_models_bigram = bigram_df.sort_values(by="Test-Acc", ascending=False).head(3)

    print("\n=== Top 3 Models (Unigram) ===")
    print(top_models_unigram[["Model", "Test-Acc"]])

    print("\n=== Top 3 Models (Bigram) ===")
    print(top_models_bigram[["Model", "Test-Acc"]])

    # Combine both top models and return them
    top_models_combined = pd.concat([top_models_unigram, top_models_bigram]).sort_values(by="Test-Acc", ascending=False).head(3)
    print("\n=== Top 3 Models Overall ===")
    print(top_models_combined[["Model", "Test-Acc"]])
    
    return best_combo


In [ ]:
# Load manually labeled data
datasets = [
    "annotated_microtext_lemma.csv",
]

polar_models = {
    "Logistic Regression": LogisticRegression(solver="lbfgs"),
    "SVM": SVC(kernel="linear"),
    "Naive Bayes": MultinomialNB(),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
}

for dataset in datasets:
    print(f"\n=== Processing Dataset: {dataset} ===")

    data = load_data(dataset)

    print(f"Loaded {len(data)} reviews from {dataset}.")
    
    # Only keep positive or negative sentiments
    data = data[data["Final Sentiment"].isin(["positive", "negative"])]
    data = data.dropna(subset=["Final Sentiment"])
    data = data.reset_index(drop=True)
    print(f"Filtered {len(data)} reviews with positive/negative sentiments.")
    print(f"Final Sentiment Counts:\n{data['Final Sentiment'].value_counts()}")
    
    polarity_detection(data,polar_models)


=== Processing Dataset: annotated_microtext_lemma.csv ===
Loaded 1249 rows from annotated_microtext_lemma.csv
Empty DataFrame
Columns: [Platform, Game, review_text, label1, label2, label3, Final Sentiment]
Index: []
Loaded 1249 reviews from annotated_microtext_lemma.csv.
Filtered 829 reviews with positive/negative sentiments.
Final Sentiment Counts:
Final Sentiment
negative    416
positive    413
Name: count, dtype: int64
Count Vectorizer Shape: (829, 7714)
Count Vectorizer Shape: (829, 48364)
TF-IDF Vectorizer Shape: (829, 7714)
TF-IDF Vectorizer Shape: (829, 48364)

=== Using model: Logistic Regression ===

--- Vectorizer: Unigram Count Vec ---

--- Vectorizer: Bigram Count Vec ---

--- Vectorizer: Unigram TF-IDF Vec ---

--- Vectorizer: Bigram TF-IDF Vec ---

=== Using model: SVM ===

--- Vectorizer: Unigram Count Vec ---

--- Vectorizer: Bigram Count Vec ---

--- Vectorizer: Unigram TF-IDF Vec ---

--- Vectorizer: Bigram TF-IDF Vec ---

=== Using model: Naive Bayes ===

--- Vector

In [ ]:

def polarity_detection_ensemble(data, models, vectorizers):
    """
    Perform polarity detection using a VotingClassifier ensemble.
    
    Parameters:
    - data: DataFrame containing 'Final Sentiment' labels.
    - models: Dictionary of {model_name: model_instance}.
    - vectorizers: Dictionary of {vectorizer_name: vectorized features}.
    - voting_type: 'hard' or 'soft' voting.
    """
    results = []
    best_score = 0.0
    best_combo = {"model_name": None, "vectorizer": None, "model": None, "accuracy": 0.0}

    # Create the VotingClassifier from the models
    estimators = [(name, model) for name, model in models.items()]
    voting_clf = VotingClassifier(estimators=estimators, voting='hard')

    for vectorizer_name, X in vectorizers.items():
        print(f"\n--- Vectorizer: {vectorizer_name} ---")

        X_train, X_test, y_train, y_test = train_test_split(
            X, data["Final Sentiment"], test_size=0.2, random_state=42
        )

        current_model = copy.deepcopy(voting_clf)

        # Cross-validation
        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        fold_accuracies, fold_precisions, fold_recalls, fold_f1_scores = [], [], [], []

        for train_idx, val_idx in kf.split(X_train):
            X_train_fold, X_val_fold = X_train[train_idx], X_train[val_idx]
            y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]

            current_model.fit(X_train_fold, y_train_fold)
            y_pred_val = current_model.predict(X_val_fold)

            fold_accuracies.append(accuracy_score(y_val_fold, y_pred_val))
            fold_precisions.append(precision_score(y_val_fold, y_pred_val, average='binary', pos_label='positive'))
            fold_recalls.append(recall_score(y_val_fold, y_pred_val, average='binary', pos_label='positive'))
            fold_f1_scores.append(f1_score(y_val_fold, y_pred_val, average='binary', pos_label='positive'))

        # Average cross-validation metrics
        k_acc, k_pre, k_rec, k_f1 = map(np.mean, (fold_accuracies, fold_precisions, fold_recalls, fold_f1_scores))

        # Train on full train set and evaluate on test set
        current_model.fit(X_train, y_train)
        y_pred_test = current_model.predict(X_test)

        test_accuracy = accuracy_score(y_test, y_pred_test)
        test_precision = precision_score(y_test, y_pred_test, average='binary', pos_label='positive')
        test_recall = recall_score(y_test, y_pred_test, average='binary', pos_label='positive')
        test_f1 = f1_score(y_test, y_pred_test, average='binary', pos_label='positive')

        results.append([
            f"VotingClassifier + {vectorizer_name}",
            k_acc, k_pre, k_rec, k_f1,
            test_accuracy, test_precision, test_recall, test_f1
        ])

        if test_accuracy > best_score:
            best_score = test_accuracy
            best_combo["model_name"] = "VotingClassifier"
            best_combo["vectorizer"] = vectorizer_name
            best_combo["model"] = current_model
            best_combo["accuracy"] = best_score

    # Final results
    columns = ["Model", "K-Acc", "K-Pre", "K-Rec", "K-F1", "Test-Acc", "Test-Pre", "Test-Rec", "Test-F1"]
    results_df = pd.DataFrame(results, columns=columns)

    print("\n=== polarity detection Results ===")
    print(results_df)

    return best_combo

In [ ]:
# Load manually labeled data
datasets = [
    "annotated_microtext_lemma.csv"
]

models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(kernel="linear"),
    "Logistic Regression": LogisticRegression(solver="lbfgs", max_iter=1000),
}

vectorizers = creating_vectorizers(data)


for dataset in datasets:
    print(f"\n=== Processing Dataset: {dataset} ===")

    data = load_data(dataset)

    print(f"Loaded {len(data)} reviews from {dataset}.")
    
    # Only keep positive or negative sentiments
    data = data[data["Final Sentiment"].isin(["positive", "negative"])]
    data = data.dropna(subset=["Final Sentiment"])
    data = data.reset_index(drop=True)
    print(f"Filtered {len(data)} reviews with positive/negative sentiments.")
    print(f"Final Sentiment Counts:\n{data['Final Sentiment'].value_counts()}")
    
    polarity_detection_ensemble(data,polar_models,vectorizers)

Count Vectorizer Shape: (829, 7714)
Count Vectorizer Shape: (829, 48364)
TF-IDF Vectorizer Shape: (829, 7714)
TF-IDF Vectorizer Shape: (829, 48364)

=== Processing Dataset: annotated_microtext_lemma.csv ===
Loaded 1249 rows from annotated_microtext_lemma.csv
Empty DataFrame
Columns: [Platform, Game, review_text, label1, label2, label3, Final Sentiment]
Index: []
Loaded 1249 reviews from annotated_microtext_lemma.csv.
Filtered 829 reviews with positive/negative sentiments.
Final Sentiment Counts:
Final Sentiment
negative    416
positive    413
Name: count, dtype: int64

--- Vectorizer: Unigram Count Vec ---

--- Vectorizer: Bigram Count Vec ---

--- Vectorizer: Unigram TF-IDF Vec ---

--- Vectorizer: Bigram TF-IDF Vec ---

=== Sentiment Analysis Results ===
                                   Model     K-Acc     K-Pre     K-Rec  \
0   VotingClassifier + Unigram Count Vec  0.728412  0.729019  0.743682   
1    VotingClassifier + Bigram Count Vec  0.734450  0.717186  0.784080   
2  VotingCl

In [48]:
def polarity_detection_gridsearch(data, param_grid):
    """
    This function performs subjectivity detection using GridSearchCV for SVC (Support Vector Classifier).
    It skips manual K-Fold and focuses only on finding the best hyperparameters.
    """

    # Assuming creating_vectorizers is defined elsewhere
    vectorizers = creating_vectorizers(data)

    best_score = 0.0
    best_combo = {"model": None, "vectorizer": None, "accuracy": 0.0, "best_params": None}
    
    for vectorizer_name, X in vectorizers.items():
        print(f"\n--- Vectorizer: {vectorizer_name} ---")
        
        X_train, X_test, y_train, y_test = train_test_split(
            X, data["Final Sentiment"], test_size=0.2, random_state=42
        )

        # Define the GridSearchCV with SVC instead of RandomForestClassifier
        grid_search = GridSearchCV(
            estimator=SVC(random_state=42),
            param_grid=param_grid,
            cv=5,  # GridSearchCV uses internal cross-validation
            n_jobs=-1,
            verbose=1
        )

        # Fit GridSearchCV
        grid_search.fit(X_train, y_train)

        best_model = grid_search.best_estimator_

        # Predict on the test set
        y_pred_test = best_model.predict(X_test)

        # Calculate test metrics
        test_accuracy = accuracy_score(y_test, y_pred_test)
        test_precision = precision_score(y_test, y_pred_test, average='binary', pos_label='positive')
        test_recall = recall_score(y_test, y_pred_test, average='binary', pos_label='positive')
        test_f1 = f1_score(y_test, y_pred_test, average='binary', pos_label='positive')

        print(f"Test Accuracy: {test_accuracy:.4f}")
        print(f"Test Precision: {test_precision:.4f}")
        print(f"Test Recall: {test_recall:.4f}")
        print(f"Test F1 Score: {test_f1:.4f}")
        print(f"Best Params: {grid_search.best_params_}")

        # Update best combo if this model is better
        if test_accuracy > best_score:
            best_score = test_accuracy
            best_combo["model"] = best_model
            best_combo["vectorizer"] = vectorizer_name
            best_combo["accuracy"] = best_score
            best_combo["best_params"] = grid_search.best_params_

    return best_combo

In [50]:
# Load manually labeled data
datasets = [
    "annotated_microtext_lemma.csv",
]

# Grid Search for Hyperparameter Tuning
param_grid = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf', 'poly'],
    'gamma': ['scale', 'auto', 0.1, 1]
}

for dataset in datasets:
    print(f"\n=== Processing Dataset: {dataset} ===")

    data = load_data(dataset)

    print(f"Loaded {len(data)} reviews from {dataset}.")
    
    # Only keep positive or negative sentiments
    data = data[data["Final Sentiment"].isin(["positive", "negative"])]
    data = data.dropna(subset=["Final Sentiment"])
    data = data.reset_index(drop=True)
    print(f"Filtered {len(data)} reviews with positive/negative sentiments.")
    print(f"Final Sentiment Counts:\n{data['Final Sentiment'].value_counts()}")
    
    # Perform Subjectivity Detection first
    polarity_detection_gridsearch(data,param_grid)



=== Processing Dataset: annotated_microtext_lemma.csv ===
Loaded 1249 rows from annotated_microtext_lemma.csv
Empty DataFrame
Columns: [Platform, Game, review_text, label1, label2, label3, Final Sentiment]
Index: []
Loaded 1249 reviews from annotated_microtext_lemma.csv.
Filtered 829 reviews with positive/negative sentiments.
Final Sentiment Counts:
Final Sentiment
negative    416
positive    413
Name: count, dtype: int64
Count Vectorizer Shape: (829, 7714)
Count Vectorizer Shape: (829, 48364)
TF-IDF Vectorizer Shape: (829, 7714)
TF-IDF Vectorizer Shape: (829, 48364)

--- Vectorizer: Unigram Count Vec ---
Fitting 5 folds for each of 48 candidates, totalling 240 fits
Test Accuracy: 0.7470
Test Precision: 0.6727
Test Recall: 0.9250
Test F1 Score: 0.7789
Best Params: {'C': 100, 'gamma': 'auto', 'kernel': 'rbf'}

--- Vectorizer: Bigram Count Vec ---
Fitting 5 folds for each of 48 candidates, totalling 240 fits
Test Accuracy: 0.7651
Test Precision: 0.7158
Test Recall: 0.8500
Test F1 Score:

## Train and Return Model

In [54]:
def train_and_return_model(model, data, target, test_size=0.2, random_state=42):
    """
    Trains the given model on the provided data and target, then returns the trained model.

    Parameters:
    model: The machine learning model (e.g., a scikit-learn model like SVM, RandomForest, etc.).
    data: The input features for training.
    target: The target labels for training.
    test_size: The proportion of data to use for testing (default is 0.2).
    random_state: Random seed for reproducibility (default is 42).

    Returns:
    trained_model: The trained machine learning model.
    """
    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(data, target, test_size=test_size, random_state=random_state)
    
    # Train the model
    model.fit(X_train, y_train)
    
    # Optionally, evaluate the model here if needed
    accuracy = model.score(X_test, y_test)
    print(f"Model Accuracy: {accuracy:.2f}")
    
    return model


In [59]:
# Subjectivity Detection
# Best Model: Logistic Regression + Unigram TF-IDF Vec     0.720

model = LogisticRegression(solver="lbfgs", max_iter=1000)

data = load_data(dataset)

unigram_tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 1)
)  # Use unigrams and bigrams
X_uni_tfidf = unigram_tfidf_vectorizer.fit_transform(data["review_text"])
print("TF-IDF Vectorizer Shape:", X_uni_tfidf.shape)

print(f"Loaded {len(data)} reviews from {dataset}.")
    
# Create a new column 'subjectivity' based on the 'Final Sentiment'
data['subjectivity'] = data['Final Sentiment'].apply(
    lambda x: 'Subjective' if x in ['positive', 'negative'] else 'Objective'
)
    
print(f"Subjectivity Detection: {data['subjectivity'].value_counts()}")

# Perform Subjectivity Detection first
trained_model = train_and_return_model(model,X_uni_tfidf,data['subjectivity'])

joblib.dump(trained_model, 'best_subjectivity_detection_model.pkl')

Loaded 1249 rows from annotated_microtext_lemma.csv
Empty DataFrame
Columns: [Platform, Game, review_text, label1, label2, label3, Final Sentiment]
Index: []
TF-IDF Vectorizer Shape: (1249, 8982)
Loaded 1249 reviews from annotated_microtext_lemma.csv.
Subjectivity Detection: subjectivity
Subjective    829
Objective     420
Name: count, dtype: int64
Model Accuracy: 0.72


['best_subjectivity_detection_model.pkl']

In [61]:
# Subjectivity Detection
# Best Model: SVM + Unigram TF-IDF Vec  0.801205

model = SVC(kernel="linear")

data = load_data(dataset)



print(f"Loaded {len(data)} reviews from {dataset}.")
    
data = data[data["Final Sentiment"].isin(["positive", "negative"])]
data = data.dropna(subset=["Final Sentiment"])
data = data.reset_index(drop=True)
print(f"Filtered {len(data)} reviews with positive/negative sentiments.")
print(f"Final Sentiment Counts:\n{data['Final Sentiment'].value_counts()}")


unigram_tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 1)
)  # Use unigrams and bigrams
X_uni_tfidf = unigram_tfidf_vectorizer.fit_transform(data["review_text"])
print("TF-IDF Vectorizer Shape:", X_uni_tfidf.shape)

# Perform Subjectivity Detection first
trained_model = train_and_return_model(model,X_uni_tfidf,data['Final Sentiment'])

joblib.dump(trained_model, 'best_polarity_detection_model.pkl')

Loaded 1249 rows from annotated_microtext_lemma.csv
Empty DataFrame
Columns: [Platform, Game, review_text, label1, label2, label3, Final Sentiment]
Index: []
Loaded 1249 reviews from annotated_microtext_lemma.csv.
Filtered 829 reviews with positive/negative sentiments.
Final Sentiment Counts:
Final Sentiment
negative    416
positive    413
Name: count, dtype: int64
TF-IDF Vectorizer Shape: (829, 7714)
Model Accuracy: 0.80


['best_polarity_detection_model.pkl']